# s01e02 - Debug Map\n\nLoads power plant locations and person sighting coordinates from the SQLite debug DB and plots them on an interactive map of Poland.

In [2]:
import sqlite3
import pandas as pd
import folium

DB_PATH = "../../outputs/s01e02_debug.db"

con = sqlite3.connect(DB_PATH)
power_plants = pd.read_sql("SELECT * FROM power_plants", con)
person_locations = pd.read_sql("SELECT * FROM person_locations", con)
person_access = pd.read_sql("SELECT * FROM person_access", con)
con.close()

print("Power plants:", len(power_plants))
print(power_plants)
print("\nPerson locations:", len(person_locations))
print(person_locations)
print("\nPerson access:", len(person_access))
print(person_access)

Power plants: 7
                   city   lat   lon       code
0                Zabrze  None  None  PWR3847PL
1  Piotrków Trybunalski  None  None  PWR5921PL
2             Grudziądz  None  None  PWR7264PL
3                 Tczew  None  None  PWR1593PL
4                 Radom  None  None  PWR8406PL
5               Chelmno  None  None  PWR2758PL
6             Żarnowiec  None  None  PWR6132PL

Person locations: 14
      name surname      lat      lon
0   Cezary   Żurek  51.1150  17.0620
1   Cezary   Żurek  50.2512  19.0125
2   Cezary   Żurek  51.6510  17.8220
3   Cezary   Żurek  50.2972  18.6610
4   Cezary   Żurek  50.6620  17.9380
5   Cezary   Żurek  50.3015  18.7912
6   Cezary   Żurek  51.7760  19.4890
7   Cezary   Żurek  51.1150  17.0620
8   Cezary   Żurek  50.2512  19.0125
9   Cezary   Żurek  51.6510  17.8220
10  Cezary   Żurek  50.2972  18.6610
11  Cezary   Żurek  50.6620  17.9380
12  Cezary   Żurek  50.3015  18.7912
13  Cezary   Żurek  51.7760  19.4890

Person access: 2
     name sur

In [3]:
# Merge access level info into person_locations for richer popups
person_locations_enriched = person_locations.merge(
    person_access[["name", "surname", "access_level"]].drop_duplicates(),
    on=["name", "surname"],
    how="left",
)

# Center map on Poland
m = folium.Map(location=[52.0, 19.5], zoom_start=6, tiles="CartoDB positron")

# Power plants - red factory icons
for _, row in power_plants.iterrows():
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue
    folium.Marker(
        location=[row["lat"], row["lon"]],
        tooltip=f"{row['city']} ({row['code']})",
        popup=folium.Popup(
            f"<b>Power plant</b><br>City: {row['city']}<br>Code: {row['code']}<br>({row['lat']:.4f}, {row['lon']:.4f})",
            max_width=220,
        ),
        icon=folium.Icon(color="red", icon="flash", prefix="fa"),
    ).add_to(m)

    # Draw a 10 km radius circle matching the proximity threshold
    folium.Circle(
        location=[row["lat"], row["lon"]],
        radius=10_000,
        color="red",
        fill=True,
        fill_opacity=0.05,
        tooltip=f"10 km radius around {row['city']}",
    ).add_to(m)

# Assign a distinct color per person
people = person_locations_enriched[["name", "surname"]].drop_duplicates()
palette = ["blue", "green", "purple", "orange", "darkred", "cadetblue", "darkgreen"]
person_colors = {
    (r["name"], r["surname"]): palette[i % len(palette)]
    for i, r in people.iterrows()
}

# Person sightings - colored circle markers
for _, row in person_locations_enriched.iterrows():
    if pd.isna(row["lat"]) or pd.isna(row["lon"]):
        continue
    color = person_colors.get((row["name"], row["surname"]), "gray")
    access_text = f"Access level: {int(row['access_level'])}" if pd.notna(row.get("access_level")) else "Access level: unknown"
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color=color,
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{row['name']} {row['surname']}",
        popup=folium.Popup(
            f"<b>{row['name']} {row['surname']}</b><br>{access_text}<br>({row['lat']:.4f}, {row['lon']:.4f})",
            max_width=220,
        ),
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
            background:white; padding:10px 14px; border-radius:6px;
            border:1px solid #ccc; font-size:13px; line-height:1.8">
  <b>Legend</b><br>
  <span style="color:red">&#9632;</span> Power plant (+ 10 km radius)<br>
"""
for (name, surname), color in person_colors.items():
    legend_html += f'  <span style="color:{color}">&#9679;</span> {name} {surname}<br>\n'
legend_html += "</div>"
m.get_root().html.add_child(folium.Element(legend_html))

m